# Period moves — outrights + curves/flies/spreads

Introduce **START** y **END**. El notebook calcula el cambio en ese intervalo y la **std de movimientos del mismo horizonte** (si el periodo ≈ 1 día → std de cambios diarios; ≈ 1 semana → semanales; etc., vía `# barras` del panel).

### Formato de fechas (elige uno)
```
10/15/26 09:00
10/15/26 9am
2026-10-15 09:00
10/15/2026 09:00
```

Listas editables en `curve_config.py`: `MOVE_OUTRIGHTS`, `MOVE_STRUCTURES`.
Spreads BBG (`USSFCT05` …): están en `CROSS_ASSET` — primera vez con `PULL_HISTORY = True`.

In [ ]:
import importlib
import logging
import os
import sys
from pathlib import Path

from IPython.display import display
import pandas as pd

logging.disable(logging.WARNING)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

_DZ = Path("/v/region/na/appl/datazone/notebooks2/data/users/arraipau/fwd_screener_v1")
REPO = Path.cwd() if (Path.cwd() / "period_moves.py").exists() else _DZ
if not (REPO / "period_moves.py").exists():
    raise FileNotFoundError("No encuentro period_moves.py — git pull")

os.chdir(REPO)
sys.path = [str(REPO)] + [p for p in sys.path if str(Path(p).resolve()) != str(REPO.resolve())]
print("REPO:", REPO.resolve())

In [ ]:
import run_dz

importlib.reload(run_dz)
run_dz.bootstrap_for_repo(str(REPO))

In [ ]:
# ── Fechas (edita aquí) ──────────────────────────────────────────────────────
START = "10/15/26 09:00"
END   = "10/16/26 12:00"

# Opcional: refrescar BBG si faltan USSFCT / tenores
PULL_HISTORY = False

import curve_config
import lib.period_moves as period_moves
from lib.data import refresh_live_store

importlib.reload(curve_config)
importlib.reload(period_moves)

# Defaults if curve_config on disk is stale (forgot git pull)
outrights_cfg = getattr(curve_config, "MOVE_OUTRIGHTS", ["2", "5", "10", "30"])
structures_cfg = getattr(
    curve_config, "MOVE_STRUCTURES",
    ["2s5s", "5s10s", "10s30s", "2s5s10s", "5s10s30s", "USSFCT05", "USSFCT10", "USSFCT30"],
)

print("START:", START, "→", period_moves.parse_ts(START))
print("END  :", END, "→", period_moves.parse_ts(END))
print("outrights :", outrights_cfg)
print("structures:", structures_cfg)
if not hasattr(curve_config, "MOVE_OUTRIGHTS"):
    print("WARNING: curve_config sin MOVE_* — haz: git pull origin main")

if PULL_HISTORY:
    refresh_live_store(force_history=True)


In [ ]:
outrights, structures, meta, panel = period_moves.run_period_moves(
    START, END, outrights=outrights_cfg, structures=structures_cfg,
)
print("horizon:", meta["horizon"])
print("bars used for std:", meta["n_bars"], "@", meta["bar_minutes"], "min")
print("nearest start/end will appear in t_start / t_end columns")


In [ ]:
# ── Tabla 1: OUTRIGHTS (2y/5y/10y/30y) en bp ─────────────────────────────────
print("OUTRIGHTS — change & std in bp  |  z = change / std(same-horizon moves)")
display(outrights)

In [ ]:
# ── Tabla 2: CURVES / FLIES / SPREADS (bp / BBG units) ───────────────────────
print("STRUCTURES — 2s5s, 5s10s, 10s30s, flies, USSFCT…")
display(structures)

In [ ]:
# Guardar CSV (opcional)
out_dir = Path("graphs/period_moves")
out_dir.mkdir(parents=True, exist_ok=True)
outrights.to_csv(out_dir / "moves_outrights.csv", index=False)
structures.to_csv(out_dir / "moves_structures.csv", index=False)
print("wrote", out_dir.resolve())